# Refining a map against the pixels

ImageD11's scanning 3DXRD pipeline fits voxel orientations and strains to peak *centroids*. Along the way it labels spots, merges them across frames and assigns them to voxels, and each step loses information: peaks from voxels along the same beam path are averaged, and centroids are snapped to frame centres.

`anri.refine.refine` instead fits the map to the measured sparse pixels themselves, with the renderer as the forward model. Every voxel's contribution to every pixel in every frame is modelled, so voxels along a beam path are fitted jointly, and the frame integration is part of the model rather than an approximation.

This notebook simulates a small grain with an orientation gradient and a strain, starts from one orientation for the whole grain (as a grain map from a tomographic reconstruction would give), and refines.

In [ ]:
import anri.utils
anri.utils.setup()

import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.spatial.transform import Rotation

import anri.crystal, anri.fwd, anri.io, anri.refine

start = time.time()

## The truth

A 5 × 5 voxel slice of an iron grain (voxels of 1 µm). Its orientation rotates by 0.01° across the grain, about a general axis, and it carries a uniform elastic strain of a few $10^{-4}$.
The map entries are the same as for the renderer: a position, a UBI and a density each.

In [ ]:
a = 2.8665
B = np.asarray(anri.crystal.lpars_to_B(np.array([a, a, a, 90.0, 90.0, 90.0])))
U0 = Rotation.from_euler("xyz", [12.0, 34.0, 56.0], degrees=True).as_matrix()

side = 5
c = np.arange(side) - (side - 1) / 2
pos = np.stack(np.meshgrid(c, c, indexing="ij"), -1).reshape(-1, 2)
pos = np.column_stack([pos, np.zeros(len(pos))])

axis = np.array([1.0, 2.0, 3.0]) / np.sqrt(14.0)
angle = np.radians(0.01) * pos[:, 0] / (side - 1)  # -0.005 to +0.005 degrees across x
R = Rotation.from_rotvec(angle[:, None] * axis).as_matrix()
strain = 1e-4 * np.array([[3.0, 1.0, 0.0], [1.0, -2.0, 0.5], [0.0, 0.5, 1.0]])
F_true = R @ (np.eye(3) + strain)  # rotation after stretch
ubi0 = np.linalg.inv(U0 @ B)
truth = {"ubi": ubi0 @ np.swapaxes(F_true, -1, -2), "pos": pos, "density": np.ones(len(pos))}

## A simulated scan

The {110} reflections on a small detector, seven dty rows covering the grain, 0.25° frames.
To keep this notebook quick, the beam divergence is larger than at a synchrotron, so peaks spread over a few frames; refinement with sharp peaks works the same way but needs a closer start (see below).

In [ ]:
hkls = np.array([[h, k, l] for h in (-1, 0, 1) for k in (-1, 0, 1) for l in (-1, 0, 1) if h * h + k * k + l * l == 2], float)
F2 = np.ones(len(hkls))
wavelength = 12.398419843320026 / 43.0
pars = {"y_center": 255.5, "y_size": 75.0, "tilt_y": 0.0, "z_center": 255.5, "z_size": 75.0, "tilt_z": 0.0,
        "tilt_x": 0.0, "distance": 60e3, "o11": -1, "o12": 0, "o21": 0, "o22": -1, "wavelength": wavelength,
        "wedge": 0.0, "chi": 0.0}
det_shape = (512, 512)
geom = anri.io.geom_from_pars(pars, 0.0, wavelength * 1e-3, 1e-3, 1e-3, sig_beam=0.5, voxel_size=1.0, sig_psf=0.7)
omega = np.arange(0.0, 180.0, 0.25) + 0.125
rows = [anri.fwd.make_row(omega, np.full_like(omega, dty)) for dty in np.arange(-3.0, 4.0)]  # the whole grain, at every angle

measured = []
for row in rows:
    frame, pixel, value, _ = anri.fwd.render_row(truth, hkls, F2, geom, row, det_shape, min_value=0.0, max_frames=7)
    measured.append(anri.refine.measured(frame, pixel, value, len(omega)))
print(sum(m["value"].size for m in measured), "measured pixels")

## The start: one orientation for the whole grain

The start gives every voxel the grain's average orientation and no strain, as a grain map from a tomographic reconstruction would.
The refinement then has to find both the orientation gradient and the strain from the pixels.

In [ ]:
first_guess = {"ubi": np.repeat(ubi0[None], len(pos), 0), "pos": pos, "density": np.ones(len(pos))}
refined, history = anri.refine.refine(first_guess, hkls, F2, geom, rows, measured, det_shape, n_iter=10, cut=0.0, max_frames=7)

plt.semilogy([h["loss"] for h in history], "o-")
plt.xlabel("iteration")
plt.ylabel("loss")
plt.show()

## How close is it?

From each refined UBI, the deformation gradient relative to the start, $F$ with $\text{UBI} = \text{UBI}_0 F^T$, splits into a rotation and a stretch (the polar decomposition $F = R\,(I + \varepsilon)$). Compare them with the truth:

In [ ]:
def rotation_and_strain(ubi):
    F = np.swapaxes(np.linalg.solve(ubi0, ubi), -1, -2)  # UBI = UBI0 F^T
    u, s, vt = np.linalg.svd(F)
    Rm = u @ vt
    eps = np.swapaxes(Rm, -1, -2) @ F - np.eye(3)
    return np.degrees(Rotation.from_matrix(Rm).as_rotvec() @ axis), 0.5 * (eps + np.swapaxes(eps, -1, -2))

ang_t, eps_t = rotation_and_strain(truth["ubi"])
ang_r, eps_r = rotation_and_strain(refined["ubi"])
fig, axs = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
axs[0].plot(pos[:, 0], ang_t, "k.", label="truth")
axs[0].plot(pos[:, 0], ang_r, "r+", label="refined")
axs[0].set(xlabel="x (µm)", ylabel="rotation about the axis (deg)", title="orientation gradient")
axs[0].legend()
for ax, (name, (i, j)) in zip(axs[1:], (("xx", (0, 0)), ("xy", (0, 1)))):
    ax.plot(pos[:, 0], 1e4 * eps_t[:, i, j], "k.", label="truth")
    ax.plot(pos[:, 0], 1e4 * eps_r[:, i, j], "r+", label="refined")
    ax.set(xlabel="x (µm)", ylabel=f"strain {name} (1e-4)", title=f"eps_{name}")
plt.show()
print(f"rotation error rms {np.sqrt(np.mean((ang_r - ang_t) ** 2)):.2e} deg; "
      f"strain error rms {1e4 * np.sqrt(np.mean((eps_r - eps_t) ** 2)):.3f} x 1e-4; "
      f"density error max {np.abs(refined['density'] - 1).max():.4f}")

## What to know before using it

- **The start must be close.** The loss is close to quadratic only within about a tenth of a peak's width. Here the peaks are broad, so a 0.005° start is fine; for sharp synchrotron peaks in 0.05° frames, the start must be within a few thousandths of a degree. Starting further away (e.g. from a grain map, where gradients can reach degrees) needs a coarse-to-fine stage that is not available yet.
- **It runs on one device** for now, one dty row at a time, so memory grows with the peaks of one row, not of the whole scan.
- **What it fits:** one deformation gradient (orientation and strain) and one density per map entry, against the pixels above the segmentation `cut`. Everything else (geometry, beam, intensity model) must be right, as for the renderer.
- **Precision:** the precision of the UBIs (float32 or float64) is the precision used. Call `anri.utils.setup()` first: on recent NVIDIA GPUs it turns off TF32 matrix products, which are far too coarse for this.

In [ ]:
print(f"Took {time.time() - start:.0f} seconds")